# MCS603 — Programming II (Python)
## Week 10–11: Web Development — Flask & Django Basics

---
### Learning Objectives
- Understand the request/response cycle and MVC/MVT pattern
- Create Flask routes, return HTML responses, and use Jinja2 templates
- Pass data from Python to templates
- Handle HTML forms with GET and POST
- Understand Django project structure (models, views, templates, URLs)

> **Setup:** `pip install flask django`

### Outline
1. Web Concepts Recap  
2. Flask — Minimal App  
3. Flask Routes & URL Parameters  
4. Jinja2 Templates  
5. Flask Forms (GET & POST)  
6. Flask Student Portal — Full Example  
7. Django Overview  
8. Django Project Structure  
9. Lab Exercises  
10. Assignment  

---
## 1. Web Concepts Recap

| Term | Meaning |
|---|---|
| **Client** | Browser (sends HTTP requests) |
| **Server** | Python app (sends HTTP responses) |
| **Route** | URL pattern mapped to a function |
| **GET** | Request data (no body) — e.g., load a page |
| **POST** | Send data to server — e.g., submit a form |
| **Template** | HTML file with placeholders filled at runtime |
| **MVC** | Model–View–Controller (architectural pattern) |
| **MVT** | Django's variant: Model–View–Template |

### Request/Response Cycle
```
Browser  ──GET /students──►  Flask App
         ◄──HTML response──  (renders template with data)
```

---
## 2. Flask — Minimal App

Flask is a **micro web framework** — minimal setup, maximum flexibility.

```python
# app.py
from flask import Flask
app = Flask(__name__)

@app.route("/")
def home():
    return "Hello, Flask!"

if __name__ == "__main__":
    app.run(debug=True)
```

Run: `python app.py` → visit `http://127.0.0.1:5000`

In [ ]:
# Write the complete Flask student portal to disk
import os
os.makedirs("student_portal/templates", exist_ok=True)

# ── app.py ──────────────────────────────────────────────────────────
app_py = '''
from flask import Flask, render_template, request, redirect, url_for

app = Flask(__name__)

# In-memory "database" for this demo
students = [
    {"id": 1, "name": "Alice",   "course": "MCS603", "grade": 92},
    {"id": 2, "name": "Bob",     "course": "MCS603", "grade": 75},
    {"id": 3, "name": "Charlie", "course": "MCS603", "grade": 88},
]
next_id = 4

@app.route("/")
def home():
    return render_template("home.html", count=len(students))

@app.route("/students")
def list_students():
    return render_template("students.html", students=students)

@app.route("/students/<int:student_id>")
def student_detail(student_id):
    student = next((s for s in students if s["id"] == student_id), None)
    if student is None:
        return "Student not found", 404
    return render_template("detail.html", student=student)

@app.route("/add", methods=["GET", "POST"])
def add_student():
    global next_id
    if request.method == "POST":
        name   = request.form["name"]
        course = request.form["course"]
        grade  = int(request.form["grade"])
        students.append({"id": next_id, "name": name,
                         "course": course, "grade": grade})
        next_id += 1
        return redirect(url_for("list_students"))
    return render_template("add.html")

if __name__ == "__main__":
    app.run(debug=True)
'''

with open("student_portal/app.py", "w") as f:
    f.write(app_py)
print("app.py written.")

In [ ]:
# ── base template ────────────────────────────────────────────────────
base_html = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Student Portal — {% block title %}{% endblock %}</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 800px;
               margin: 40px auto; padding: 0 20px; }
        nav a { margin-right: 15px; text-decoration: none;
                color: #0066cc; }
        table { border-collapse: collapse; width: 100%; }
        th, td { border: 1px solid #ccc; padding: 8px 12px; text-align: left; }
        th { background: #f0f0f0; }
        .btn { background: #0066cc; color: white; padding: 8px 16px;
               border: none; cursor: pointer; border-radius: 4px; }
        input, select { padding: 6px; margin: 4px 0; width: 100%; }
    </style>
</head>
<body>
<nav>
    <a href="/">Home</a>
    <a href="/students">Students</a>
    <a href="/add">Add Student</a>
</nav>
<hr>
{% block content %}{% endblock %}
</body>
</html>
'''

home_html = '''{% extends "base.html" %}
{% block title %}Home{% endblock %}
{% block content %}
<h1>Student Portal</h1>
<p>Welcome! There are currently <strong>{{ count }}</strong> students enrolled.</p>
<a href="/students">View All Students</a>
{% endblock %}
'''

students_html = '''{% extends "base.html" %}
{% block title %}Students{% endblock %}
{% block content %}
<h2>Enrolled Students</h2>
<table>
    <tr><th>ID</th><th>Name</th><th>Course</th><th>Grade</th><th>Action</th></tr>
    {% for s in students %}
    <tr>
        <td>{{ s.id }}</td>
        <td>{{ s.name }}</td>
        <td>{{ s.course }}</td>
        <td>{{ s.grade }}</td>
        <td><a href="/students/{{ s.id }}">Details</a></td>
    </tr>
    {% else %}
    <tr><td colspan="5">No students yet.</td></tr>
    {% endfor %}
</table>
{% endblock %}
'''

detail_html = '''{% extends "base.html" %}
{% block title %}{{ student.name }}{% endblock %}
{% block content %}
<h2>{{ student.name }}</h2>
<p><strong>ID:</strong> {{ student.id }}</p>
<p><strong>Course:</strong> {{ student.course }}</p>
<p><strong>Grade:</strong> {{ student.grade }}/100</p>
<p><strong>Status:</strong>
{% if student.grade >= 50 %}
    <span style="color:green">PASS</span>
{% else %}
    <span style="color:red">FAIL</span>
{% endif %}
</p>
<a href="/students">← Back to list</a>
{% endblock %}
'''

add_html = '''{% extends "base.html" %}
{% block title %}Add Student{% endblock %}
{% block content %}
<h2>Add New Student</h2>
<form method="POST">
    <label>Name:<br><input type="text" name="name" required></label>
    <label>Course:<br><input type="text" name="course" value="MCS603"></label>
    <label>Grade (0–100):<br>
        <input type="number" name="grade" min="0" max="100" required>
    </label>
    <br><br>
    <button class="btn" type="submit">Add Student</button>
</form>
{% endblock %}
'''

for fname, content in [
    ("base.html",     base_html),
    ("home.html",     home_html),
    ("students.html", students_html),
    ("detail.html",   detail_html),
    ("add.html",      add_html),
]:
    with open(f"student_portal/templates/{fname}", "w") as f:
        f.write(content)

print("All templates written.")
print("\nTo run:  cd student_portal && python app.py")
print("Then visit: http://127.0.0.1:5000")

---
## 3. Flask Routes & URL Parameters

```python
@app.route("/students/<int:student_id>")
def detail(student_id):
    ...
```

| Converter | Type | Example URL |
|---|---|---|
| `<int:x>` | integer | `/students/42` |
| `<float:x>` | float | `/price/3.14` |
| `<string:x>` | string (default) | `/user/alice` |
| `<path:x>` | path with `/` | `/files/a/b/c` |

### Query String Parameters
`/search?q=alice&page=2` → `request.args.get("q")`

---
## 4. Jinja2 Templates — Key Syntax

| Syntax | Purpose |
|---|---|
| `{{ variable }}` | Output a value |
| `{% if condition %}` | Conditional block |
| `{% for item in list %}` | Loop |
| `{% extends "base.html" %}` | Inherit from parent template |
| `{% block name %}` | Define/override a named block |
| `{{ value \| upper }}` | Apply a filter |

### Common Filters
| Filter | Effect |
|---|---|
| `upper` / `lower` | Case |
| `length` | Number of items |
| `round(2)` | Round float |
| `default("N/A")` | Fallback for undefined |

---
## 5. Flask Forms — GET vs POST

```python
@app.route("/add", methods=["GET", "POST"])
def add():
    if request.method == "POST":
        name = request.form["name"]     # form field
        q    = request.args.get("q")    # query string
        return redirect(url_for("list"))
    return render_template("add.html")
```

| Data source | Access |
|---|---|
| Form (POST body) | `request.form["field"]` |
| Query string (GET) | `request.args.get("key")` |
| URL parameter | Function argument |
| JSON body | `request.get_json()` |

---
## 6. Django Overview

Django is a **full-featured, batteries-included framework** for larger applications.

| | Flask | Django |
|---|---|---|
| Size | Micro | Full-stack |
| ORM | Manual (SQLAlchemy) | Built-in |
| Admin panel | No | Auto-generated |
| Auth system | No | Built-in |
| Best for | APIs, small apps | Large web apps |

### MVT Pattern
```
URL dispatcher → View → Model (database)
                  ↓
               Template
```

---
## 7. Django Project Structure

```bash
django-admin startproject mcs603_site
cd mcs603_site
python manage.py startapp students
```

```
mcs603_site/
├── manage.py                ← CLI tool
├── mcs603_site/
│   ├── settings.py          ← project config (DB, apps, ...)
│   ├── urls.py              ← root URL config
│   └── wsgi.py              ← deployment entry point
└── students/
    ├── models.py            ← database table definitions
    ├── views.py             ← request handlers
    ├── urls.py              ← app-level URL patterns
    ├── admin.py             ← register models for admin UI
    └── templates/students/  ← HTML templates
```

In [ ]:
# Write illustrative Django files (study material — not a runnable project here)
import os
os.makedirs("django_demo/students", exist_ok=True)

# models.py
with open("django_demo/students/models.py", "w") as f:
    f.write('''
from django.db import models

class Student(models.Model):
    name       = models.CharField(max_length=100)
    student_id = models.CharField(max_length=20, unique=True)
    course     = models.CharField(max_length=50)
    grade      = models.IntegerField(default=0)
    enrolled   = models.DateField(auto_now_add=True)

    def __str__(self):
        return f"{self.name} ({self.student_id})"

    def letter_grade(self):
        if self.grade >= 90: return "A"
        if self.grade >= 80: return "B"
        if self.grade >= 70: return "C"
        if self.grade >= 60: return "D"
        return "F"

    class Meta:
        ordering = ["name"]
''')

# views.py
with open("django_demo/students/views.py", "w") as f:
    f.write('''
from django.shortcuts import render, get_object_or_404
from .models import Student

def student_list(request):
    students = Student.objects.all()
    return render(request, "students/list.html", {"students": students})

def student_detail(request, pk):
    student = get_object_or_404(Student, pk=pk)
    return render(request, "students/detail.html", {"student": student})

def student_create(request):
    if request.method == "POST":
        Student.objects.create(
            name=request.POST["name"],
            student_id=request.POST["student_id"],
            course=request.POST["course"],
            grade=int(request.POST["grade"]),
        )
        return redirect("student_list")
    return render(request, "students/create.html")
''')

# urls.py
with open("django_demo/students/urls.py", "w") as f:
    f.write('''
from django.urls import path
from . import views

urlpatterns = [
    path("",         views.student_list,   name="student_list"),
    path("<int:pk>/", views.student_detail, name="student_detail"),
    path("add/",     views.student_create, name="student_create"),
]
''')

print("Django demo files written to django_demo/")
print("\nTo create a real project:")
print("  pip install django")
print("  django-admin startproject mcs603_site")
print("  cd mcs603_site && python manage.py startapp students")
print("  python manage.py makemigrations && python manage.py migrate")
print("  python manage.py runserver")

---
## 8. Django ORM — Quick Reference

```python
# Create
s = Student.objects.create(name="Alice", grade=90)

# Read all
Student.objects.all()

# Filter
Student.objects.filter(grade__gte=70)

# Get single record (raises 404 if missing)
get_object_or_404(Student, pk=1)

# Update
Student.objects.filter(pk=1).update(grade=95)

# Delete
Student.objects.filter(pk=1).delete()

# Order
Student.objects.order_by("-grade")   # descending

# Count
Student.objects.count()
```

---
## 9. Lab Exercises

### Lab 1: Extend the Flask Student Portal
Add the following routes to `student_portal/app.py`:

1. `GET /students/search?q=<name>` — filter students by name (partial match)
2. `POST /students/<id>/delete` — remove a student from the list
3. `GET /stats` — show class average, highest grade, lowest grade

Create corresponding templates using the existing `base.html`.

In [ ]:
# Write extended routes to student_portal/app.py
extra_routes = '''
@app.route("/students/search")
def search():
    q       = request.args.get("q", "").lower()
    results = [s for s in students if q in s["name"].lower()]
    return render_template("search.html", results=results, query=q)

@app.route("/students/<int:sid>/delete", methods=["POST"])
def delete_student(sid):
    global students
    students = [s for s in students if s["id"] != sid]
    return redirect(url_for("list_students"))

@app.route("/stats")
def stats():
    if not students:
        return render_template("stats.html", stats=None)
    grades = [s["grade"] for s in students]
    data   = {"avg": sum(grades)/len(grades),
               "high": max(grades), "low": min(grades),
               "count": len(grades)}
    return render_template("stats.html", stats=data)
'''
print(extra_routes)

### Lab 2: Explore Django Project Structure
Answer the following by reading the files written to `django_demo/`:

1. What does `models.Meta.ordering` do?
2. What does `get_object_or_404` do that `Student.objects.get(pk=pk)` doesn't?
3. How would you add a `email` field to the Student model?
4. What change is needed in `urls.py` to add a route for editing a student?

**Your answers here:**

1. 
2. 
3. 
4. 

---
## 10. Assignment — Week 10–11

**Build a Complete Flask Web Application**

Create a **Course Registration System** with the following pages:

| Route | Description |
|---|---|
| `GET /` | Home page with summary stats |
| `GET /courses` | List all courses with student counts |
| `GET /students` | List all students with search filter |
| `GET /students/<id>` | Student profile with their courses |
| `POST /students/add` | Add a student |
| `POST /enroll` | Enrol a student in a course |
| `GET /stats` | Class statistics per course |

**Data model** (in-memory dicts/lists):  
- `students`: `{id, name, email}`  
- `courses`: `{code, title, max_capacity}`  
- `enrollments`: `{student_id, course_code, grade}`

**Requirements:**
- All pages inherit from `base.html`
- Forms use POST with redirect-after-POST pattern
- Input validation with user-friendly error messages
- At least one page uses a Jinja2 loop AND conditional

In [ ]:
# Plan your routes and data structures here before implementing in app.py


---
## Summary — Week 10–11

| Concept | Key Point |
|---|---|
| Flask route | `@app.route("/path")` maps URL → function |
| URL parameters | `<int:id>` in route; function receives as argument |
| Templates | `render_template("file.html", key=value)` |
| Jinja2 | `{{ var }}`, `{% if %}`, `{% for %}`, `{% extends %}` |
| Forms | `request.form` (POST), `request.args` (GET query) |
| Redirect | `redirect(url_for("view_name"))` after POST |
| Django | MVT, ORM, auto-admin, `manage.py` CLI |

**Next:** Week 12–13 — Database Access (SQLite / MySQL)